# Step 4: Initial Data Diagnostics & Temporal Schema Alignment

### 1 - HIGH-LEVEL OVERVIEW

* **The Pipeline Phase:** We are entering the Exploratory Data Analysis (EDA) & Data Preprocessing stage of our machine learning pipeline.
* **The Problematics:** Real-world datasets often load with unoptimized or incorrect data types (e.g., dates read as generic text strings, integers loaded as floats). If left unaddressed, text-based dates prevent us from performing chronological calculations, and heavy, default data types consume excessive memory, slowing down down-stream model training.
* **Our Solution:** In this step, we establish a robust computational baseline. We import our core analytical libraries, ingest the prepared dataset, explicitly cast the raw arrival timeline into a true datetime schema, and run a structural diagnostic check to audit the data types and row volume.

In [ ]:
import pandas as pd
import numpy as np
import holidays

# Load the project dataset into a pandas DataFrame structure
df = pd.read_csv('ml_ready_bookings.csv')

# Explicitly cast the arrival date column from a generic object/string 
# datatype into a specialized datetime64 object to enable time-series operations
df['arrival_date'] = pd.to_datetime(df['arrival_date'])

# In order to run a first check on the dataset, in particular on the data types, 
# we printed the info about the dataset alongside the total row volume
print(f"Total rows: {df.shape[0]}")
print(df.info())

#### Code Logic & Architecture Explanations:
* **`import holidays`**: Brings in the specialized holiday tracking library, which we will use to map country-specific statutory calendars directly to our checkout timelines without relying on fragile, manual date-mapping tables.
* **`pd.to_datetime()`**: Converts the string representation of dates (e.g., `"2026-07-16"`) into a datetime object. This unlocks access to pandas attributes like `.dt.year`, `.dt.dayofweek`, and seamless chronological filtering.
* **`df.shape[0]`**: Programmatically counts the number of rows along the vertical axis (axis 0) of the DataFrame, giving us an instant confirmation of the data volume we are manipulating.
* **`df.info()`**: Prints a comprehensive summary of the DataFrame, including index data types, non-null counts per column, and memory usage. This is our primary diagnostic tool to check for missing values and optimize memory allocation before feeding data into a machine learning model.

### 2 - MEMORY OPTIMIZATION & TYPE INVARIANCE ENFORCEMENT

* **The Pipeline Phase:** Memory Optimization & Type Invariance Enforcement.
* **The Problematics:** By default, Pandas reads numeric columns containing missing values (NaNs) as heavy float64 values, which unnecessarily inflates the memory footprint of ID fields like 'agent' and 'company'. Furthermore, monetary metrics like Average Daily Rate (ADR) do not require double-precision float64 storage, wasting system RAM during large-scale matrix operations.
* **Our Solution:** We align the rest of our time-series schemas, convert identifier columns into Pandas' native nullable Integer format ('Int64') to elegantly handle missing records without altering data types, and downcast monetary columns to single-precision float32 to reduce RAM strain.

In [ ]:
# 1. Convert booking_date to datetime to match the arrival timeline schema
df['booking_date'] = pd.to_datetime(df['booking_date'])

# 2. Convert ID columns to Nullable Integers (capital 'I' Int64) to allow NaNs 
# without forcing the entire column to become a float datatype
df['agent'] = df['agent'].astype('Int64')
df['company'] = df['company'].astype('Int64')

# 3. Downcast monetary columns to single-precision float32 to optimize memory efficiency
df['adr'] = df['adr'].astype('float32')
df['total_hotel_revenue_on_arrival_date'] = df['total_hotel_revenue_on_arrival_date'].astype('float32')

# Print structural info again to audit the optimized changes and verify memory reduction
print(df.info())

#### Code Logic & Architecture Explanations:
* **`.astype('Int64')`**: Notice the capital **I**. Unlike standard NumPy integers, Pandas' native `Int64` supports missing values (`NaN`). This prevents an ID column with blank cells from being forcefully cast into a float, preserving the integer property of hotel identifiers.
* **`.astype('float32')`**: Downcasts values from the standard 64-bit float to a 32-bit float. Since the numbers in revenue columns don't require scientific-level precision, cutting the bit-allocation in half reduces the dataset's memory footprint dramatically without losing financial accuracy.

### 3.1 - GLOBAL CALENDAR INITIALIZATION & ARRAY MAPPING

We construct three mathematically independent reference arrays (Sets) to map out our distinct demand periods.
*   **Static School Arrays:** We hardcode the exact, pre-shifted historical school breaks for Portugal (2015-2017) to capture actual family check-in behavior.
*   **Dynamic Statutory Arrays:** We instantiate the Portuguese calendar and apply asymmetric weekday gravity rules (e.g., Tuesday holidays extending back to Friday). Dates already swallowed by school holidays are explicitly stripped out to prevent feature collision.

In [ ]:
import pandas as pd
import datetime
import holidays

# 1. MAP STATIC SCHOOL ARRAYS (Pre-shifted for actual check-in behavior)

# 1A. Summer Breaks
summer_ranges = [
    pd.date_range(start='2015-07-01', end='2015-09-14'),
    pd.date_range(start='2016-06-08', end='2016-09-11'),
    pd.date_range(start='2017-06-15', end='2017-08-31')
]
summer_dates = set([date.date() for drange in summer_ranges for date in drange])

# 1B. Other School Breaks (Christmas, Carnival, Easter)
other_school_ranges = [
    pd.date_range(start='2015-12-16', end='2016-01-03'), # Christmas '15
    pd.date_range(start='2016-12-16', end='2017-01-01'), # Christmas '16
    pd.date_range(start='2016-02-05', end='2016-02-09'), # Carnival '16
    pd.date_range(start='2017-02-24', end='2017-02-28'), # Carnival '17
    pd.date_range(start='2016-03-18', end='2016-04-03'), # Easter '16
    pd.date_range(start='2017-04-04', end='2017-04-16')  # Easter '17
]
other_school_dates = set([date.date() for drange in other_school_ranges for date in drange])


# 2. GENERATE DYNAMIC STATUTORY ARRAYS (Weekend Bridges)

pt_holidays = holidays.Portugal(years=[2015, 2016, 2017])
bridge_dates_list = []

for holiday_date, name in pt_holidays.items():
    weekday = holiday_date.weekday() # 0=Mon, 1=Tue, 2=Wed, 3=Thu, 4=Fri, 5=Sat, 6=Sun
    
    if weekday == 0:     # Monday Holiday -> Friday to Monday
        deltas = [-3, -2, -1, 0]
    elif weekday == 1:   # Tuesday Holiday -> Friday to Tuesday
        deltas = [-4, -3, -2, -1, 0]
    elif weekday == 2:   # Wednesday Holiday -> Preceding Saturday to Sunday
        deltas = [-4, -3]
    elif weekday == 3:   # Thursday Holiday -> Wednesday to Sunday
        deltas = [-1, 0, 1, 2, 3]
    elif weekday == 4:   # Friday Holiday -> Thursday to Sunday
        deltas = [-1, 0, 1, 2]
    else:                # Weekend Holiday -> No extension
        deltas = [0]
        
    for delta in deltas:
        bridge_dates_list.append(holiday_date + datetime.timedelta(days=delta))

# 3. ISOLATE THE SETS (Ensure zero feature collision)
weekend_bridge_dates = set(bridge_dates_list)
# Subtract dates already handled by the macro school blocks
weekend_bridge_dates = weekend_bridge_dates - summer_dates - other_school_dates

print("--- CALENDAR MAPPING SUCCESS ---")
print(f"Summer Holiday Dates: {len(summer_dates)}")
print(f"Other School Holiday Dates: {len(other_school_dates)}")
print(f"Independent Statutory Bridge Dates: {len(weekend_bridge_dates)}")

### 3.2 - RESERVATION FOOTPRINT EXPANSION

We transition from evaluating single check-in dates to evaluating the entire reservation duration. By adding the length of stay to the arrival date, we generate an exact array of dates representing every single night the guest is physically staying at the property.

In [ ]:
# 1. CALCULATE CHECKOUT DATE
# Sum the weekend and week nights to determine total duration
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# Add total duration to arrival_date to get the exact departure date
df['departure_date'] = df['arrival_date'] + pd.to_timedelta(df['total_nights'], unit='D')

# 2. EXPAND THE RANGE (RESERVATION FOOTPRINT)
# Generate a sequence of physical calendar dates for the guest's entire stay
def generate_stay_sequence(row):
    # Handle day-use reservations (0 nights) by assigning just the arrival date
    if row['total_nights'] == 0:
        return [row['arrival_date'].date()]
    
    # Generate the date sequence from check-in up to the night before check-out
    return [(row['arrival_date'] + pd.Timedelta(days=i)).date() for i in range(row['total_nights'])]

# Apply the function to create the footprint array for every single row
df['stay_date_sequence'] = df.apply(generate_stay_sequence, axis=1)

# Print data audit to verify the expansion visually
print("--- FOOTPRINT EXPANSION SUCCESS ---")
print(df[['arrival_date', 'total_nights', 'departure_date', 'stay_date_sequence']].head())

### 3.3 - THE BINARY INTERSECTION ENGINE

With our holiday arrays and guest footprints built, we evaluate overlaps using Python's high-speed set intersections. Each of the three columns is populated with a binary indicator (`1` or `0`) indicating whether the reservation footprint collides with that specific holiday category. This preserves independent, overlapping signals for our machine learning classifiers.

In [ ]:
# 1. DEFINE THE INTERSECTION LOGIC
# We use Python's native set intersection for maximum computational speed
def evaluate_overlap(stay_sequence, holiday_set):
    # If there is any overlapping date between the stay and the holiday reference set, return 1
    return 1 if set(stay_sequence).intersection(holiday_set) else 0

# 2. EVALUATE OVERLAPS FOR THE 3-COLUMN ARCHITECTURE
# Evaluate Summer
df['summer_holiday_stay_type'] = df['stay_date_sequence'].apply(
    lambda seq: evaluate_overlap(seq, summer_dates)
)

# Evaluate Other School
df['other_school_holiday_stay_type'] = df['stay_date_sequence'].apply(
    lambda seq: evaluate_overlap(seq, other_school_dates)
)

# Evaluate Weekend Bridges
df['weekend_bridge_stay_type'] = df['stay_date_sequence'].apply(
    lambda seq: evaluate_overlap(seq, weekend_bridge_dates)
)

print("--- BINARY INTERSECTION ENGINE SUCCESS ---")
print(df[['arrival_date', 'summer_holiday_stay_type', 'other_school_holiday_stay_type', 'weekend_bridge_stay_type']].head())

### 3.4 - MATRIX CLEANUP & OVERLAP VALIDATION

The temporary object arrays generated during expansion consume substantial memory. We drop these intermediate features to restore our optimized RAM baseline. Finally, we run a diagnostic audit to verify that the independent overlap architecture is functioning properly by isolating reservations that fall into multiple domains at once.

In [ ]:
# 1. MATRIX CLEANUP (DROP HEAVY VARIABLES)
# Drop temporary lists and intermediate columns to free up RAM
df = df.drop(columns=['stay_date_sequence', 'total_nights'])

# 2. AUDIT OUTPUT & OVERLAP VALIDATION
# Compute a temporary concurrency score to locate reservations spanning multiple categories
df['concurrent_flags'] = df['summer_holiday_stay_type'] + df['other_school_holiday_stay_type'] + df['weekend_bridge_stay_type']
overlapping_bookings = df[df['concurrent_flags'] >= 2]

# Drop temporary flag helper
df = df.drop(columns=['concurrent_flags'])

print("--- MATRIX CLEANUP & OVERLAP AUDIT SUCCESS ---")
print(f"Total bookings spanning multiple holiday domains simultaneously: {len(overlapping_bookings):,}")
if len(overlapping_bookings) > 0:
    print("\nSample of overlapping multi-flagged reservations:")
    print(overlapping_bookings[['arrival_date', 'departure_date', 'summer_holiday_stay_type', 'other_school_holiday_stay_type', 'weekend_bridge_stay_type']].head(10))

### 5.1 - FEATURE PIPELINE ENGINEERING & CATEGORICAL ENCODING

Before initializing model training, we establish the algorithmic feature matrix ($X$) and label vector ($y$). We handle structural gaps and prevent data leakage via an automated blueprint:
*   **Missing Value Allocation:** Missing categorical countries are isolated into an explicit `'Unknown'` string category. Missing numeric identifiers for agents or companies are imputed with a constant sentinel flag (`-1`) to map structural absence cleanly.
*   **One-Hot Encoding:** High-dimensional categorical text parameters (`hotel`, `deposit_type`, `market_segment`, `meal`, `customer_type`, `distribution_channel`, `reserved_room_type`) are encoded into independent numeric dummy bit-flags.
*   **Chronological Guardrails:** The absolute datetime stamps (`booking_date`, `arrival_date`, `departure_date`) are dropped entirely from the training matrix. This isolates the tree-split mechanics to focus completely on your dynamic pace proxies, seasonal markers, and holiday profiles.

In [ ]:
# 1. DEFINE TARGET & ISOLATE CATEGORICAL / NUMERIC BOUNDARIES
target_variable = 'is_canceled'
y = df[target_variable].copy()

# Explicitly drop the target and raw time anchors from the feature space to prevent chronological data leak
features_to_drop = [target_variable, 'booking_date', 'arrival_date', 'departure_date']
X_raw = df.drop(columns=features_to_drop)

# 2. EXPLICIT MISSING VALUE IMPUTATION
# Handle missing text entities by creating an isolated structural 'Unknown' string dimension
if 'country' in X_raw.columns:
    X_raw['country'] = X_raw['country'].fillna('Unknown')
if 'distribution_channel' in X_raw.columns:
    X_raw['distribution_channel'] = X_raw['distribution_channel'].fillna('Unknown')

# Handle missing nullable integer identifiers using a numeric sentinel flag (-1)
if 'agent' in X_raw.columns:
    X_raw['agent'] = X_raw['agent'].fillna(-1).astype('int64')
if 'company' in X_raw.columns:
    X_raw['company'] = X_raw['company'].fillna(-1).astype('int64')

# 3. STABLE ONE-HOT ENCODING FOR CATEGORICAL DIMENSIONS
categorical_columns = [
    'hotel', 'deposit_type', 'market_segment', 'meal', 
    'customer_type', 'distribution_channel', 'reserved_room_type', 'country'
 ]

# Automatically extract features and construct numeric dummy arrays
X = pd.get_dummies(X_raw, columns=categorical_columns, drop_first=True)

# Convert all remaining boolean columns or dummy arrays into clean integers
bool_cols = X.select_dtypes(include=['bool']).columns
X[bool_cols] = X[bool_cols].astype(int)

print("--- FEATURE SELECTION & PREPROCESSING PIPELINE SUCCESS ---")
print(f"Target Array Shape (y): {y.shape}")
print(f"Encoded Feature Matrix Shape (X): {X.shape}")
print(f"Total Numeric Columns Prepared for ML Ingestion: {len(X.columns)}")